In [ ]:
import io

import pandas as pd
import plotly.express as px
import streamlit as st

st.set_page_config(page_title="Dashboard Cukai", page_icon="📊", layout="wide")

# ---------- Column names ----------
COL_MONTH = "BULAN"
COL_COMPANY = "NAMA SYARIKAT"
COL_VIOLATION = "JENIS PELANGGARAN SYARAT"
COL_CODE = "KOD HASIL"
COL_AMOUNT = "JUMLAH CUKAI"
COL_DATE = "TARIKH BAYARAN"
REQUIRED = [COL_MONTH, COL_COMPANY, COL_VIOLATION, COL_CODE, COL_AMOUNT, COL_DATE]

MONTHS = {
    "JANUARI": 1, "JAN": 1, "JANUARY": 1,
    "FEBRUARI": 2, "FEB": 2, "FEBRUARY": 2,
    "MAC": 3, "MAR": 3, "MARCH": 3,
    "APRIL": 4, "APR": 4,
    "MEI": 5, "MAY": 5,
    "JUN": 6, "JUNE": 6,
    "JULAI": 7, "JUL": 7, "JULY": 7,
    "OGOS": 8, "OGO": 8, "AUG": 8, "AUGUST": 8,
    "SEPTEMBER": 9, "SEP": 9, "SEPT": 9,
    "OKTOBER": 10, "OKT": 10, "OCT": 10, "OCTOBER": 10,
    "NOVEMBER": 11, "NOV": 11,
    "DISEMBER": 12, "DIS": 12, "DEC": 12, "DECEMBER": 12,
}


# ---------- Helpers ----------
@st.cache_data(show_spinner=False)
def load_csv(file_bytes: bytes) -> pd.DataFrame:
    df = None
    for enc in ("utf-8-sig", "latin-1"):
        try:
            df = pd.read_csv(io.BytesIO(file_bytes), sep=None, engine="python", encoding=enc)
            break
        except UnicodeDecodeError:
            continue
    if df is None:
        raise ValueError("Could not read the file. Please save it as a CSV (UTF-8).")
    # Normalise headers: strip stray tabs/spaces, collapse double spaces, uppercase
    df.columns = [" ".join(str(c).split()).upper() for c in df.columns]
    return df


def clean_amount(series: pd.Series) -> pd.Series:
    cleaned = series.astype(str).str.replace(r"[^\d.\-]", "", regex=True)
    return pd.to_numeric(cleaned, errors="coerce")


def month_number(value):
    if pd.isna(value):
        return None
    text = str(value).strip().upper()
    if text in MONTHS:
        return MONTHS[text]
    try:
        n = int(float(text))
        if 1 <= n <= 12:
            return n
    except ValueError:
        pass
    return None


def build_period_date(df: pd.DataFrame, basis: str) -> pd.Series:
    """Return the first day of the month for each row, based on the chosen date column."""
    pay_date = pd.to_datetime(df[COL_DATE], dayfirst=True, errors="coerce")
    if basis == COL_DATE:
        return pay_date.dt.to_period("M").dt.to_timestamp()

    # BULAN basis: month name/number + year taken from TARIKH BAYARAN
    months = pd.to_numeric(df[COL_MONTH].map(month_number), errors="coerce")
    combined = pd.to_datetime(
        pd.DataFrame({"year": pay_date.dt.year, "month": months, "day": 1}),
        errors="coerce",
    )
    # If BULAN already holds a full date such as 2024-03 or 03/2024
    direct = pd.to_datetime(df[COL_MONTH], dayfirst=True, errors="coerce")
    direct = direct.where(months.isna()).dt.to_period("M").dt.to_timestamp()
    return combined.fillna(direct)


def rm(x: float) -> str:
    return f"RM {x:,.2f}"


# ---------- Sidebar: upload ----------
st.title("📊 Dashboard Cukai")
st.sidebar.header("1. Muat naik data")
uploaded = st.sidebar.file_uploader("Upload CSV", type=["csv"])

if uploaded is None:
    st.info("Upload a CSV file in the sidebar to generate the dashboard.")
    st.markdown("**Expected columns:** " + ", ".join(f"`{c}`" for c in REQUIRED))
    st.stop()

try:
    raw = load_csv(uploaded.getvalue())
except Exception as e:
    st.error(f"Failed to read the file: {e}")
    st.stop()

missing = [c for c in REQUIRED if c not in raw.columns]
if missing:
    st.error(f"Missing column(s): {', '.join(missing)}")
    st.write("Columns found in your file:", list(raw.columns))
    st.stop()

# ---------- Sidebar: view options ----------
st.sidebar.header("2. Paparan")
view = st.sidebar.radio("Dashboard view", ["Bulanan (Monthly)", "Tahunan (Yearly)"])
basis = st.sidebar.selectbox(
    "Build time axis from",
    [COL_DATE, COL_MONTH],
    help="TARIKH BAYARAN is used by default. Switch to BULAN if that column defines your reporting month.",
)
top_n = st.sidebar.slider("Top N companies / categories", 5, 30, 10)

# ---------- Prepare data ----------
df = raw.copy()
df[COL_AMOUNT] = clean_amount(df[COL_AMOUNT])
df["PERIOD"] = build_period_date(df, basis)
df["TAHUN"] = df["PERIOD"].dt.year

bad_dates = int(df["PERIOD"].isna().sum())
bad_amounts = int(df[COL_AMOUNT].isna().sum())
if bad_dates or bad_amounts:
    st.warning(
        f"{bad_dates} row(s) have an unreadable date and {bad_amounts} row(s) have an "
        f"unreadable amount. Rows without a date are excluded from the time charts."
    )
df[COL_AMOUNT] = df[COL_AMOUNT].fillna(0)

# ---------- Sidebar: filters ----------
st.sidebar.header("3. Tapis (leave empty = all)")
years = sorted(df["TAHUN"].dropna().astype(int).unique())
sel_years = st.sidebar.multiselect("Tahun", years)
sel_companies = st.sidebar.multiselect("Nama Syarikat", sorted(df[COL_COMPANY].dropna().unique()))
sel_violations = st.sidebar.multiselect("Jenis Pelanggaran Syarat", sorted(df[COL_VIOLATION].dropna().unique()))
sel_codes = st.sidebar.multiselect("Kod Hasil", sorted(df[COL_CODE].dropna().astype(str).unique()))

f = df.copy()
f[COL_CODE] = f[COL_CODE].astype(str)
if sel_years:
    f = f[f["TAHUN"].isin(sel_years)]
if sel_companies:
    f = f[f[COL_COMPANY].isin(sel_companies)]
if sel_violations:
    f = f[f[COL_VIOLATION].isin(sel_violations)]
if sel_codes:
    f = f[f[COL_CODE].isin(sel_codes)]

if f.empty:
    st.warning("No data matches the current filters.")
    st.stop()

# ---------- Aggregate by period ----------
dated = f.dropna(subset=["PERIOD"])
if view.startswith("Bulanan"):
    trend = dated.groupby("PERIOD", as_index=False).agg(
        JUMLAH=(COL_AMOUNT, "sum"), REKOD=(COL_AMOUNT, "size")
    )
    trend["LABEL"] = trend["PERIOD"].dt.strftime("%b %Y")
    period_name = "Bulan"
else:
    trend = dated.groupby("TAHUN", as_index=False).agg(
        JUMLAH=(COL_AMOUNT, "sum"), REKOD=(COL_AMOUNT, "size")
    )
    trend["LABEL"] = trend["TAHUN"].astype(int).astype(str)
    period_name = "Tahun"

# ---------- KPIs ----------
total = f[COL_AMOUNT].sum()
k1, k2, k3, k4 = st.columns(4)
k1.metric("Jumlah Cukai", rm(total))
k2.metric("Bilangan Rekod", f"{len(f):,}")
k3.metric("Bilangan Syarikat", f"{f[COL_COMPANY].nunique():,}")
k4.metric("Purata / Rekod", rm(total / len(f)))

if not trend.empty:
    peak = trend.loc[trend["JUMLAH"].idxmax()]
    st.caption(f"Highest {period_name.lower()}: **{peak['LABEL']}** with {rm(peak['JUMLAH'])}")

st.divider()

# ---------- Trend ----------
st.subheader(f"Jumlah Cukai mengikut {period_name}")
if trend.empty:
    st.info("No rows with a valid date to plot.")
else:
    if view.startswith("Bulanan"):
        fig = px.line(trend, x="PERIOD", y="JUMLAH", markers=True,
                      labels={"PERIOD": "Bulan", "JUMLAH": "Jumlah Cukai (RM)"})
    else:
        fig = px.bar(trend, x="LABEL", y="JUMLAH", text_auto=".3s",
                     labels={"LABEL": "Tahun", "JUMLAH": "Jumlah Cukai (RM)"})
        fig.update_xaxes(type="category")
    st.plotly_chart(fig, use_container_width=True)

    fig2 = px.bar(trend, x="LABEL", y="REKOD",
                  labels={"LABEL": period_name, "REKOD": "Bilangan Rekod"},
                  title=f"Bilangan rekod mengikut {period_name.lower()}")
    fig2.update_xaxes(type="category")
    st.plotly_chart(fig2, use_container_width=True)

# ---------- Breakdowns ----------
c1, c2 = st.columns(2)

with c1:
    st.subheader(f"Top {top_n} Syarikat")
    comp = (f.groupby(COL_COMPANY, as_index=False)[COL_AMOUNT].sum()
            .nlargest(top_n, COL_AMOUNT).sort_values(COL_AMOUNT))
    st.plotly_chart(
        px.bar(comp, x=COL_AMOUNT, y=COL_COMPANY, orientation="h",
               labels={COL_AMOUNT: "Jumlah Cukai (RM)"}),
        use_container_width=True,
    )

with c2:
    st.subheader("Jenis Pelanggaran Syarat")
    vio = (f.groupby(COL_VIOLATION, as_index=False)[COL_AMOUNT].sum()
           .nlargest(top_n, COL_AMOUNT).sort_values(COL_AMOUNT))
    st.plotly_chart(
        px.bar(vio, x=COL_AMOUNT, y=COL_VIOLATION, orientation="h",
               labels={COL_AMOUNT: "Jumlah Cukai (RM)"}),
        use_container_width=True,
    )

c3, c4 = st.columns(2)

with c3:
    st.subheader("Kod Hasil")
    code = f.groupby(COL_CODE, as_index=False)[COL_AMOUNT].sum().nlargest(top_n, COL_AMOUNT)
    st.plotly_chart(
        px.pie(code, names=COL_CODE, values=COL_AMOUNT, hole=0.4),
        use_container_width=True,
    )

with c4:
    st.subheader("Pelanggaran Syarat mengikut Tempoh")
    if dated.empty:
        st.info("No dated rows.")
    else:
        key = "PERIOD" if view.startswith("Bulanan") else "TAHUN"
        stack = dated.groupby([key, COL_VIOLATION], as_index=False)[COL_AMOUNT].sum()
        figs = px.bar(stack, x=key, y=COL_AMOUNT, color=COL_VIOLATION,
                      labels={COL_AMOUNT: "Jumlah Cukai (RM)", key: period_name})
        if key == "TAHUN":
            figs.update_xaxes(type="category")
        st.plotly_chart(figs, use_container_width=True)

# ---------- Table & download ----------
st.divider()
st.subheader("Summary table")
st.dataframe(trend.drop(columns=["PERIOD"], errors="ignore"), use_container_width=True)

with st.expander("View filtered raw data"):
    st.dataframe(f.drop(columns=["PERIOD"]), use_container_width=True)

st.download_button(
    "⬇️ Download filtered data (CSV)",
    f.drop(columns=["PERIOD"]).to_csv(index=False).encode("utf-8-sig"),
    file_name="data_ditapis.csv",
    mime="text/csv",
)